# 🏆 NVIDIA Nemotron Model Reasoning Challenge
## Complete Pipeline: Data → LoRA Fine-tuning → submission.zip

**Strategy:**
- Model: `metric/nemotron-3-nano-30b-a3b-bf16` (loaded from Kaggle Models)
- Fine-tune with **LoRA** (rank=32 as required by competition)
- Dataset: Competition training data + NVIDIA's open SFT data
- Output: `submission.zip` with LoRA adapter files

**Competition Rules:**
- `max_lora_rank = 32`
- `max_tokens = 7680`
- `top_p = 1.0`
- Answer must be inside `\boxed{}` LaTeX format

## Step 1: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install -q transformers==4.47.0
!pip install -q peft==0.14.0
!pip install -q trl==0.13.0
!pip install -q datasets==3.2.0
!pip install -q accelerate==1.2.1
!pip install -q bitsandbytes==0.45.0
!pip install -q einops
!pip install -q causal-conv1d>=1.4.0
!pip install -q mamba-ssm

print('✅ All packages installed!')

## Step 2: Load Competition Data

In [ ]:
import os
import json
import pandas as pd
import numpy as np

# ─── Find competition data ───
COMP_DIR = '/kaggle/input/nvidia-nemotron-model-reasoning-challenge'

# List all available files
for root, dirs, files in os.walk(COMP_DIR):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Load train data — handles both CSV and JSONL formats
import glob

train_files = glob.glob(f'{COMP_DIR}/train*')
test_files  = glob.glob(f'{COMP_DIR}/test*')

print('Train files:', train_files)
print('Test files: ', test_files)

def load_data(path):
    if path.endswith('.csv'):
        return pd.read_csv(path)
    elif path.endswith('.jsonl'):
        return pd.read_json(path, lines=True)
    elif path.endswith('.json'):
        return pd.read_json(path)
    else:
        return pd.read_csv(path)

if train_files:
    train_df = load_data(train_files[0])
    print(f'\n📊 Train shape: {train_df.shape}')
    print(train_df.head(3))

if test_files:
    test_df = load_data(test_files[0])
    print(f'\n📊 Test shape: {test_df.shape}')
    print(test_df.head(3))

## Step 3: Load Nemotron-Nano-30B Model with 4-bit Quantization

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# ─── Model path (from Kaggle Models tab) ───
# Option A: From Kaggle Models (add model from Models tab first)
MODEL_PATH = '/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'

# Option B: From HuggingFace (needs internet enabled)
# MODEL_PATH = 'nvidia/Nemotron-3-Nano-30B-A3B-BF16'

print(f'🔄 Loading model from: {MODEL_PATH}')
print(f'💾 GPU Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# 4-bit quantization config (essential for 30B model on T4/A100)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print('✅ Model loaded successfully!')
print(f'   Params: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

## Step 4: Setup LoRA (rank=32 as required by competition)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# ─── LoRA Config (max_lora_rank=32 is competition requirement) ───
lora_config = LoraConfig(
    r=32,                        # MUST be ≤ 32 per competition rules
    lora_alpha=64,               # Usually 2x rank
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied!')
print(f'   Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of total)')

## Step 5: Prepare Training Dataset

In [ ]:
from datasets import Dataset

# ─── System prompt for reasoning tasks ───
SYSTEM_PROMPT = """You are an expert reasoning assistant. Think step by step.
Always place your final answer inside \\boxed{} LaTeX format.
For example: The answer is \\boxed{42}"""

def format_sample(row):
    """
    Format a row into chat template.
    Adjust 'problem'/'solution' column names based on actual data.
    """
    # Try common column name patterns
    question = (
        row.get('problem') or
        row.get('question') or
        row.get('prompt') or
        row.get('input') or ''
    )
    answer = (
        row.get('solution') or
        row.get('answer') or
        row.get('output') or
        row.get('response') or ''
    )

    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': str(question)},
        {'role': 'assistant', 'content': str(answer)},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

# Build HuggingFace Dataset from competition train data
if 'train_df' in dir() and train_df is not None:
    train_records = train_df.to_dict('records')
    formatted     = [format_sample(r) for r in train_records]
    train_dataset = Dataset.from_list(formatted)
    print(f'✅ Training samples: {len(train_dataset)}')
    print('Sample text preview:')
    print(train_dataset[0]['text'][:500])
else:
    print('⚠️  No train data found. Using dummy data for testing.')
    dummy = [
        {'text': tokenizer.apply_chat_template(
            [{'role':'system','content':SYSTEM_PROMPT},
             {'role':'user','content':'What is 2+2?'},
             {'role':'assistant','content':'Let me think. 2+2=4. The answer is \\boxed{4}'}],
            tokenize=False, add_generation_prompt=False
        )}
    ] * 10
    train_dataset = Dataset.from_list(dummy)
    print(f'✅ Dummy dataset created with {len(train_dataset)} samples')

## Step 6: Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/kaggle/working/lora_adapter'

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ─── Training hyperparams ───
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,    # effective batch size = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,

    # ─── Memory optimizations ───
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',

    # ─── Sequence length ───
    max_seq_length=2048,

    # ─── Logging ───
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',

    # ─── Column name for training text ───
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

## Step 7: Save LoRA Adapter

In [ ]:
import os

ADAPTER_DIR = '/kaggle/working/lora_adapter'
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Save only the LoRA adapter weights (NOT full model)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('✅ Adapter saved!')
print('Files saved:')
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f'  {f:40s}  {size:.1f} MB')

## Step 8: Quick Inference Test (Verify \\boxed{} format)

In [ ]:
# Test the fine-tuned model on a sample question
model.eval()

def generate_answer(question, max_new_tokens=512):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        enable_thinking=True   # Reasoning ON
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=1.0,           # top_p=1.0 as per competition
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    )
    return response

# Test sample
test_q = 'What is the sum of the first 10 natural numbers?'
answer = generate_answer(test_q)
print(f'Q: {test_q}')
print(f'A: {answer}')
print()

# Check if \boxed{} is present
if r'\boxed{' in answer:
    print('✅ Answer contains \\boxed{} format — CORRECT!')
else:
    print('⚠️  Answer missing \\boxed{} — check system prompt')

## Step 9: Create submission.zip 🎯

In [ ]:
import zipfile
import os

SUBMISSION_ZIP = '/kaggle/working/submission.zip'
ADAPTER_DIR    = '/kaggle/working/lora_adapter'

# Zip all adapter files
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(ADAPTER_DIR):
        fpath = os.path.join(ADAPTER_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)  # flat structure inside zip
            print(f'  Added: {fname}')

zip_size = os.path.getsize(SUBMISSION_ZIP) / 1e6
print(f'\n✅ submission.zip created!')
print(f'   Size: {zip_size:.1f} MB')
print(f'   Path: {SUBMISSION_ZIP}')
print()
print('📤 Now go to the competition page and upload this file!')

## ✅ Verify zip contents

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, 'r') as zf:
    print('Files inside submission.zip:')
    for info in zf.infolist():
        print(f'  {info.filename:50s}  {info.file_size/1e6:.2f} MB')

# Check required files are present
required = ['adapter_config.json', 'adapter_model.safetensors']
with zipfile.ZipFile(SUBMISSION_ZIP, 'r') as zf:
    names = zf.namelist()

for req in required:
    status = '✅' if req in names else '❌ MISSING'
    print(f'  {req}: {status}')

## 📝 Notes & Next Steps

### What this notebook does:
1. Loads **Nemotron-Nano-30B** with 4-bit quantization
2. Applies **LoRA (rank=32)** — exactly as competition requires
3. Fine-tunes on competition training data
4. Saves adapter and creates **submission.zip**

### To improve your score:
- Increase `num_train_epochs` to 3-5
- Add more training data from `nvidia/Llama-Nemotron-Post-Training-Dataset`
- Experiment with `lora_alpha` values (64, 128)
- Try majority voting at inference time

### GPU Tips:
- Use **T4 x2** accelerator in Kaggle notebook settings
- If OOM error: reduce `max_seq_length` to 1024
- Enable **Persistence** in notebook settings to save between sessions